In [3]:
import os
from datetime import datetime
import pytz
import pandas as pd
from supabase import create_client
from dotenv import  load_dotenv

In [4]:
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

In [5]:
supabase = create_client(SUPABASE_URL,SUPABASE_KEY)

In [6]:
def current_ist_date():
    ist = pytz.timezone("Asia/Kolkata")
    return datetime.now(ist).strftime("%Y-%m-%d")

In [7]:
print(current_ist_date())

2026-03-05


In [8]:
def create_classroom(class_name,code="1234",daily_limit=10):
    existing = (
        supabase.table("classroom_setting")
        .select("*")
        .eq("class_name",class_name)
        .execute()
        .data
    )

    if existing:
        print(f"Class '{class_name}' already exists.")
        return
    
    supabase.table("classroom_setting").insert({
        "class_name" : class_name,
        "code" : code,
        "daily_limit" : daily_limit,
        "is_open" : True
    }).execute()

    print(f"Classroom '{class_name}' created successfully")


In [9]:
create_classroom("Demo_class2",code="4264",daily_limit=5)

Classroom 'Demo_class2' created successfully


In [11]:
def add_student(class_name,roll_number,name):
    existing = (
        supabase.table("roll_map")
        .select("*")
        .eq("roll_number",roll_number)
        .execute()
        .data
    )

    if existing:
        print(f"Roll {roll_number} already assigned to {existing[0]['name']}")
        return
    

    supabase.table("roll_map").insert({
        "class_name" : class_name,
        "roll_number": roll_number,
        "name" : name
    }).execute()

    print(f"Added {name} (Roll {roll_number}) to {class_name}")

In [12]:
add_student("Demo_class2","1","saurav")
add_student("Demo_class2","2","lokesh")
add_student("Demo_class2","3","dipesh")

Added saurav (Roll 1) to Demo_class2
Added lokesh (Roll 2) to Demo_class2
Added dipesh (Roll 3) to Demo_class2


In [13]:
add_student("Demo_class2","1","sam")

Roll 1 already assigned to saurav


In [14]:
def mark_attendance(class_name,roll_number,code):
    today = current_ist_date()

    settings = (
        supabase.table("classroom_setting")
        .select("*")
        .eq("class_name",class_name)
        .execute()
        .data[0]
    )

    if code != settings["code"]:
        print("Incorrect code entered.")
        return
    
    existing = (
        supabase.table("attendance")
        .select("*")
        .eq("class_name" , class_name)
        .eq("roll_number",roll_number)
        .eq("date",today)
        .execute()
        .data
    )


    if existing:
        print("Attendance already marked today.")
        return
    

    count_today = (
        supabase.table("attendance")
        .select("*",count="exact")
        .eq("class_name",class_name)
        .eq("date", today)
        .execute()
        .count
    )

    if count_today >= settings["daily_limit"]:
        print("Daily attendance limit reached.")
        return
    

    student = (
        supabase.table("roll_map")
        .select("*")
        .eq("class_name",class_name)
        .eq("roll_number" , roll_number)
        .execute()
        .data
    )

    if not student:
        print("Roll number not registered")
        return
    

    name = student[0]["name"]

    supabase.table("attendance").insert({
        "class_name" : class_name,
        "roll_number": roll_number,
        "name": name,
        "date" : today
    }).execute()


    print(f"Attendance recorded for {name} ({roll_number})")

In [16]:
mark_attendance("Demo_class2","1","4264")
mark_attendance("Demo_class2","2","4264")
mark_attendance("Demo_class2","3","4264")

Attendance already marked today.
Attendance recorded for lokesh (2)
Attendance recorded for dipesh (3)
